# RAG Day 5 — Complete Google Colab Flow (Free / No OpenAI)

This notebook follows the original Day 5 structure:
- native document ingestion
- sensible overlapping chunks
- local 384-dimensional embeddings
- Chroma vector database
- 2D and 3D t-SNE visualization
- reranking
- query rewriting
- RAG answer generation
- retrieval evaluation
- answer evaluation

Everything runs without OpenAI, OpenAI API keys, `litellm`, or OpenAI models.

**Important:** because this is a new Colab runtime, the first flow uploads and extracts:
`evaluation.zip`, `implementation.zip`, and `knowledge-base.zip`.

In [1]:
# 1. Install required packages.
# No OpenAI or LiteLLM.

!pip install -q \
    chromadb \
    sentence-transformers \
    transformers \
    accelerate \
    scikit-learn \
    plotly \
    pydantic \
    tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 125.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [2]:
# 2. Upload the project ZIP files.
# In a fresh Colab runtime, run this cell and select:
#   evaluation.zip
#   implementation.zip
#   knowledge-base.zip

from google.colab import files
from pathlib import Path

required_zips = {
    "evaluation.zip",
    "implementation.zip",
    "knowledge-base.zip",
}

existing = {
    p.name
    for p in Path("/content").glob("*.zip")
}

missing = required_zips - existing

if missing:
    print("Upload these files:", sorted(missing))
    files.upload()
else:
    print("All required ZIP files are already in /content.")

Upload these files: ['evaluation.zip', 'implementation.zip', 'knowledge-base.zip']


Saving evaluation.zip to evaluation.zip
Saving implementation.zip to implementation.zip
Saving knowledge-base.zip to knowledge-base.zip


In [3]:
# 3. Extract the ZIP files safely.

import zipfile
import shutil
from pathlib import Path

for folder_name in [
    "evaluation",
    "implementation",
    "knowledge-base",
]:
    folder = Path("/content") / folder_name

    if folder.exists():
        shutil.rmtree(folder)

for zip_name in [
    "evaluation.zip",
    "implementation.zip",
    "knowledge-base.zip",
]:
    zip_path = Path("/content") / zip_name

    if not zip_path.exists():
        raise FileNotFoundError(
            f"{zip_name} is missing. Upload it first."
        )

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/content")

print("Content folders:")
print([
    p.name
    for p in Path("/content").iterdir()
    if p.is_dir()
])

Content folders:
['.config', 'evaluation', 'implementation', 'knowledge-base', 'sample_data']


In [4]:
# 4. Verify the knowledge base and project files.

KB = Path("/content/knowledge-base")
EVAL = Path("/content/evaluation")
IMPL = Path("/content/implementation")

md_files = list(KB.rglob("*.md"))

print("Knowledge-base files:", len(md_files))
print("Evaluation folder:", EVAL.exists())
print("Implementation folder:", IMPL.exists())

if len(md_files) == 0:
    raise RuntimeError(
        "No Markdown files found in /content/knowledge-base."
    )

if not EVAL.exists():
    raise RuntimeError("evaluation folder was not extracted.")

if not IMPL.exists():
    raise RuntimeError("implementation folder was not extracted.")

Knowledge-base files: 76
Evaluation folder: True
Implementation folder: True


In [5]:
# 5. Make the evaluation/implementation folders importable.

import sys
import importlib

(EVAL / "__init__.py").touch()
(IMPL / "__init__.py").touch()

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

importlib.invalidate_caches()

print("Python project path ready.")

Python project path ready.


In [6]:
# 6. Load the original evaluation test questions.

from evaluation.test import TestQuestion, load_tests

tests = load_tests()

print("Evaluation tests:", len(tests))

if tests:
    example = tests[0]
    print("\nExample question:")
    print(example.question)
    print("\nKeywords:")
    print(example.keywords)
    print("\nReference answer:")
    print(example.reference_answer)

Evaluation tests: 150

Example question:
Who won the prestigious IIOTY award in 2023?

Keywords:
['Maxine', 'Thompson', 'IIOTY']

Reference answer:
Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023.


In [7]:
# 7. Configuration.

import os
import re
import math
import numpy as np

from pathlib import Path
from pydantic import BaseModel, Field

KNOWLEDGE_BASE_PATH = Path("/content/knowledge-base")

DB_NAME = "/content/preprocessed_db_free"
COLLECTION_NAME = "docs"

# Free local embedding model.
# This produces 384 dimensions.
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Free local generation model.
GENERATION_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

AVERAGE_CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
RETRIEVAL_K = 10

print("Embedding:", EMBEDDING_MODEL)
print("Generation:", GENERATION_MODEL)

Embedding: sentence-transformers/all-MiniLM-L6-v2
Generation: Qwen/Qwen2.5-0.5B-Instruct


In [8]:
# 8. Native document loader, matching the original Day 5 approach.

def fetch_documents():
    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():

        if not folder.is_dir():
            continue

        doc_type = folder.name

        for file in folder.rglob("*.md"):

            with open(file, "r", encoding="utf-8") as f:
                text = f.read()

            documents.append({
                "type": doc_type,
                "source": file.as_posix(),
                "text": text,
            })

    return documents


documents = fetch_documents()

print("Loaded documents:", len(documents))

if not documents:
    raise RuntimeError("No documents loaded.")

Loaded documents: 76


In [9]:
!pip install -q langchain-text-splitters

In [10]:
# 9. Chunk the documents.
#
# Original Day 5 used an LLM to choose chunk boundaries.
# We replace that API-dependent step with a deterministic local splitter.
# This preserves the complete source text and overlap.

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=AVERAGE_CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

class Result(BaseModel):
    page_content: str
    metadata: dict


def create_chunks(documents):

    chunks = []

    for document in documents:

        pieces = splitter.split_text(
            document["text"]
        )

        for index, piece in enumerate(pieces):

            if not piece.strip():
                continue

            words = piece.strip().split()

            headline = " ".join(words[:12])

            if len(words) > 12:
                headline += "..."

            summary = (
                f"Chunk {index + 1} from "
                f"{document['type']}."
            )

            chunks.append(
                Result(
                    page_content=(
                        headline
                        + "\n\n"
                        + summary
                        + "\n\n"
                        + piece
                    ),
                    metadata={
                        "source": document["source"],
                        "type": document["type"],
                        "chunk": index + 1,
                    },
                )
            )

    return chunks


chunks = create_chunks(documents)

print("Chunks:", len(chunks))

if not chunks:
    raise RuntimeError("No chunks were created.")

Chunks: 884


In [11]:
# 10. Create local 384-dimensional embeddings.

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)

texts = [
    chunk.page_content
    for chunk in chunks
]

vectors = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

vectors = np.asarray(
    vectors,
    dtype=np.float32
)

print("Vectors:", len(vectors))
print("Dimensions:", vectors.shape[1])

assert vectors.shape[1] == 384

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Vectors: 884
Dimensions: 384


In [12]:
# 11. Create a fresh Chroma database.
#
# Removing the old directory avoids the readonly database error
# from earlier Colab sessions.

import chromadb

if os.path.exists(DB_NAME):
    shutil.rmtree(DB_NAME)

chroma = chromadb.PersistentClient(
    path=DB_NAME
)

collection = chroma.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

ids = [
    str(i)
    for i in range(len(chunks))
]

metadatas = [
    chunk.metadata
    for chunk in chunks
]

collection.add(
    ids=ids,
    embeddings=vectors.tolist(),
    documents=texts,
    metadatas=metadatas,
)

print("Vectors in Chroma:", collection.count())

assert collection.count() == len(chunks)

Vectors in Chroma: 884


In [13]:
# 12. Inspect the vector store.

stored = collection.get(
    limit=1,
    include=["embeddings", "documents", "metadatas"]
)

print(
    "Vector dimensions:",
    len(stored["embeddings"][0])
)

Vector dimensions: 384


In [14]:
# 13. 2D t-SNE visualization.

from sklearn.manifold import TSNE
import plotly.graph_objects as go

all_data = collection.get(
    include=[
        "embeddings",
        "documents",
        "metadatas",
    ]
)

plot_vectors = np.asarray(
    all_data["embeddings"],
    dtype=np.float32
)

plot_docs = all_data["documents"]
plot_metas = all_data["metadatas"]

types = [
    m.get("type", "unknown")
    for m in plot_metas
]

type_names = sorted(set(types))
type_numbers = {
    name: i
    for i, name in enumerate(type_names)
}

labels = [
    type_numbers[t]
    for t in types
]

perplexity = min(
    30,
    max(5, (len(plot_vectors) - 1) // 3)
)

reduced_2d = TSNE(
    n_components=2,
    random_state=42,
    perplexity=perplexity,
).fit_transform(plot_vectors)

fig = go.Figure(
    data=[
        go.Scatter(
            x=reduced_2d[:, 0],
            y=reduced_2d[:, 1],
            mode="markers",
            marker=dict(
                size=5,
                color=labels,
                opacity=0.8,
            ),
            text=[
                f"Type: {t}<br>{d[:150]}..."
                for t, d in zip(types, plot_docs)
            ],
            hoverinfo="text",
        )
    ]
)

fig.update_layout(
    title="2D Chroma Vector Store Visualization",
    xaxis_title="x",
    yaxis_title="y",
    width=800,
    height=600,
)

fig.show()

In [15]:
# 14. 3D t-SNE visualization.

reduced_3d = TSNE(
    n_components=3,
    random_state=42,
    perplexity=perplexity,
).fit_transform(plot_vectors)

fig3 = go.Figure(
    data=[
        go.Scatter3d(
            x=reduced_3d[:, 0],
            y=reduced_3d[:, 1],
            z=reduced_3d[:, 2],
            mode="markers",
            marker=dict(
                size=4,
                color=labels,
                opacity=0.8,
            ),
            text=[
                f"Type: {t}<br>{d[:150]}..."
                for t, d in zip(types, plot_docs)
            ],
            hoverinfo="text",
        )
    ]
)

fig3.update_layout(
    title="3D Chroma Vector Store Visualization",
    width=900,
    height=700,
)

fig3.show()

## Advanced RAG — Retrieval + Reranking + Query Rewriting

The original notebook used `gpt-4.1-nano` for reranking and query rewriting. Those calls have been replaced with local deterministic methods so the same pipeline works without an API key.

In [16]:
# 15. Local vector retrieval.

def embed_query(question):
    return embedding_model.encode(
        [question],
        normalize_embeddings=True,
    )[0].tolist()


def fetch_context_unranked(
    question,
    k=RETRIEVAL_K,
):

    query_vector = embed_query(
        question
    )

    results = collection.query(
        query_embeddings=[query_vector],
        n_results=k,
        include=[
            "documents",
            "metadatas",
            "distances",
        ],
    )

    retrieved = []

    for document, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):

        retrieved.append(
            Result(
                page_content=document,
                metadata={
                    **metadata,
                    "distance": float(distance),
                },
            )
        )

    return retrieved

In [17]:
# 16. Local reranking.
#
# Score = 70% vector similarity + 30% keyword overlap.
# No LLM/API call.

def tokenize(text):
    return set(
        re.findall(
            r"\b[a-zA-Z0-9]{2,}\b",
            text.lower(),
        )
    )


def rerank(question, chunks):

    question_terms = tokenize(question)

    scored = []

    for index, chunk in enumerate(chunks):

        distance = float(
            chunk.metadata.get(
                "distance",
                1.0,
            )
        )

        vector_score = 1.0 / (
            1.0 + max(distance, 0.0)
        )

        chunk_terms = tokenize(
            chunk.page_content
        )

        overlap = (
            len(question_terms & chunk_terms)
            / len(question_terms)
            if question_terms
            else 0.0
        )

        score = (
            0.70 * vector_score
            + 0.30 * overlap
        )

        scored.append(
            (score, index, chunk)
        )

    scored.sort(
        key=lambda x: x[0],
        reverse=True,
    )

    return [
        item[2]
        for item in scored
    ]

In [18]:
# 17. Query rewriting.
#
# This is deliberately lightweight and local.
# It keeps the user's important terms and resolves short follow-ups
# using the previous user question.

def rewrite_query(
    question,
    history=None,
):

    history = history or []

    words = question.strip().split()

    if len(words) <= 4:

        previous_questions = [
            m.get("content", "")
            for m in history
            if m.get("role") == "user"
        ]

        if previous_questions:

            return (
                previous_questions[-1]
                + " "
                + question
            ).strip()

    return question.strip()

In [19]:
# 18. Test retrieval + reranking.

question = "Who won the IIOTY award?"

query = rewrite_query(
    question,
    [],
)

unranked = fetch_context_unranked(
    query,
    k=10,
)

ranked = rerank(
    query,
    unranked,
)

print("Retrieved:", len(unranked))

for i, chunk in enumerate(
    ranked[:5],
    start=1,
):

    print(f"\n--- Rank {i} ---")
    print(chunk.page_content[:500])

Retrieved: 10

--- Rank 1 ---
## Other HR Notes - Maxine participated in various company-sponsored trainings related...

Chunk 10 from employees.

## Other HR Notes
- Maxine participated in various company-sponsored trainings related to big data technologies and cloud infrastructure.  
- She was recognized for her contributions with the prestigious Insurellm IIOTY Innovator Award in 2023.  
- Maxine is currently involved in the women-in-tech initiative and participates in mentorship programs to guide junior employees.

--- Rank 2 ---
- **Awards**: - Insurellm "SDR of the Year" Award (2022) - Monthly...

Chunk 7 from employees.

- **Awards**:  
  - Insurellm "SDR of the Year" Award (2022)  
  - Monthly MVP Recognition (3 times in 2023)  

- **Interests**:  
  - In Alex's spare time, they enjoy participating in community volunteer programs, particularly those focused on financial literacy.  
  - Alex is also an avid runner and has participated in several charity marathons.

--- Rank 3 --

In [20]:
# 19. Load the free local generation model.

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
)

tokenizer = AutoTokenizer.from_pretrained(
    GENERATION_MODEL
)

generation_model = AutoModelForCausalLM.from_pretrained(
    GENERATION_MODEL
)

generator = pipeline(
    "text-generation",
    model=generation_model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    do_sample=False,
    return_full_text=False,
)

print("Generation model loaded.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Generation model loaded.


In [21]:
# 20. RAG prompt.

SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.

Answer the user's question using only the provided knowledge-base context.

If the context does not contain the answer, say:
"I don't know based on the available knowledge base."

Be accurate, relevant, and concise.

Knowledge-base context:
{context}
"""


def make_rag_prompt(
    question,
    history,
    chunks,
):

    context = "\n\n".join(
        (
            f"Source: "
            f"{chunk.metadata.get('source', 'unknown')}\n"
            f"{chunk.page_content}"
        )
        for chunk in chunks
    )

    history_text = "\n".join(
        (
            f"{m.get('role', 'user')}: "
            f"{m.get('content', '')}"
        )
        for m in (history or [])
    )

    return f"""
{SYSTEM_PROMPT.format(context=context)}

Conversation history:
{history_text}

User question:
{question}

Answer:
""".strip()

In [22]:
# 21. Complete free Advanced RAG function.

def answer_question(
    question,
    history=None,
):

    history = history or []

    query = rewrite_query(
        question,
        history,
    )

    unranked = fetch_context_unranked(
        query,
        k=RETRIEVAL_K,
    )

    ranked = rerank(
        query,
        unranked,
    )

    context_chunks = ranked[:RETRIEVAL_K]

    prompt = make_rag_prompt(
        question,
        history,
        context_chunks,
    )

    output = generator(
        prompt
    )

    answer = output[0][
        "generated_text"
    ].strip()

    return answer, context_chunks

In [23]:
# 22. Test the RAG system.

answer, retrieved = answer_question(
    "Who won the prestigious IIOTY award in 2023?"
)

print("ANSWER:")
print(answer)

print("\nRETRIEVED:", len(retrieved))

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


ANSWER:
Insurellm "SDR of the Year" Award (2022)
Monthly MVP Recognition (3 times in 2023)

I don't know based on the available knowledge base.

RETRIEVED: 10


In [24]:
# 23. Test the second original Day 5 question.

answer, retrieved = answer_question(
    "Who went to Manchester University?"
)

print("ANSWER:")
print(answer)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER:
The user asked about Jessica Liu, who is a senior UX/UI designer at DesignStudio NYC. The context mentions that she graduated from University of Manchester. Therefore, the answer is:

University of Manchester
I don't know based on the available knowledge base.


In [53]:
# 24. Retrieval evaluation
# ------------------------------------------------------------

class RetrievalEval(BaseModel):
    mrr: float
    ndcg: float
    keywords_found: int
    total_keywords: int
    keyword_coverage: float


def get_doc_text(doc):
    """
    Convert different document/result formats into plain text.
    Works with LangChain Documents, custom Result objects,
    and plain strings.
    """
    if hasattr(doc, "page_content"):
        return doc.page_content

    if hasattr(doc, "text"):
        return doc.text

    return str(doc)


def calculate_mrr(keyword, retrieved_docs):
    """
    Mean Reciprocal Rank for one keyword.
    Returns 1/rank of the first document containing the keyword.
    """
    keyword = keyword.lower().strip()

    for rank, doc in enumerate(retrieved_docs, start=1):

        text = get_doc_text(doc).lower()

        if keyword in text:
            return 1.0 / rank

    return 0.0


def calculate_dcg(relevances, k):
    """
    Calculate Discounted Cumulative Gain.
    """
    return sum(
        relevances[i] / math.log2(i + 2)
        for i in range(
            min(k, len(relevances))
        )
    )


def calculate_ndcg(keyword, retrieved_docs, k=10):
    """
    Calculate nDCG for one keyword.
    """
    keyword = keyword.lower().strip()

    relevances = []

    for doc in retrieved_docs[:k]:

        text = get_doc_text(doc).lower()

        relevances.append(
            1 if keyword in text else 0
        )

    dcg = calculate_dcg(
        relevances,
        k,
    )

    ideal = sorted(
        relevances,
        reverse=True,
    )

    idcg = calculate_dcg(
        ideal,
        k,
    )

    return (
        dcg / idcg
        if idcg
        else 0.0
    )


def evaluate_retrieval(test, k=10, candidate_k=50):

    # --------------------------------------------------------
    # Step 1: Retrieve a larger candidate pool
    # --------------------------------------------------------

    candidates = fetch_context_unranked(
        test.question,
        k=candidate_k,
    )

    # --------------------------------------------------------
    # Step 2: Rerank the larger candidate pool
    # --------------------------------------------------------

    retrieved = rerank(
        test.question,
        candidates,
    )

    # --------------------------------------------------------
    # Step 3: Keep only the final top-k documents
    # --------------------------------------------------------

    retrieved = retrieved[:k]

    # --------------------------------------------------------
    # Step 4: Calculate MRR for every keyword
    # --------------------------------------------------------

    mrr_scores = [
        calculate_mrr(
            keyword,
            retrieved,
        )
        for keyword in test.keywords
    ]

    # --------------------------------------------------------
    # Step 5: Calculate nDCG for every keyword
    # --------------------------------------------------------

    ndcg_scores = [
        calculate_ndcg(
            keyword,
            retrieved,
            k,
        )
        for keyword in test.keywords
    ]

    # --------------------------------------------------------
    # Step 6: Determine which keywords were found
    # --------------------------------------------------------

    found = sum(
        score > 0
        for score in mrr_scores
    )

    total = len(test.keywords)

    # --------------------------------------------------------
    # Step 7: Return evaluation result
    # --------------------------------------------------------

    return RetrievalEval(
        mrr=(
            sum(mrr_scores) / len(mrr_scores)
            if mrr_scores
            else 0.0
        ),

        ndcg=(
            sum(ndcg_scores) / len(ndcg_scores)
            if ndcg_scores
            else 0.0
        ),

        keywords_found=found,

        total_keywords=total,

        keyword_coverage=(
            found / total
            if total
            else 0.0
        ),
    )


# ------------------------------------------------------------
# 25. Evaluate the first test
# ------------------------------------------------------------

if tests:

    retrieval_result = evaluate_retrieval(
        tests[0],
        k=10,
        candidate_k=50,
    )

    print(retrieval_result)

    # --------------------------------------------------------
    # Show exactly which keywords were found
    # --------------------------------------------------------

    test = tests[0]

    candidates = fetch_context_unranked(
        test.question,
        k=50,
    )

    retrieved = rerank(
        test.question,
        candidates,
    )[:10]

    print("\nQuestion:")
    print(test.question)

    print("\nKeyword Results:")

    for keyword in test.keywords:

        found = False
        rank_found = None

        for rank, doc in enumerate(
            retrieved,
            start=1,
        ):

            text = get_doc_text(doc).lower()

            if keyword.lower() in text:

                found = True
                rank_found = rank
                break

        if found:
            print(
                f"  {keyword}: FOUND "
                f"(rank {rank_found})"
            )
        else:
            print(
                f"  {keyword}: NOT FOUND"
            )

    # --------------------------------------------------------
    # Show final retrieved documents
    # --------------------------------------------------------

    print("\nTop Retrieved Documents:")

    for i, doc in enumerate(
        retrieved,
        start=1,
    ):

        print(
            f"\n--- Result {i} ---"
        )

        print(
            get_doc_text(doc)[:500]
        )

mrr=0.6666666666666666 ndcg=0.6666666666666666 keywords_found=2 total_keywords=3 keyword_coverage=0.6666666666666666

Question:
Who won the prestigious IIOTY award in 2023?

Keyword Results:
  Maxine: FOUND (rank 1)
  Thompson: NOT FOUND
  IIOTY: FOUND (rank 1)

Top Retrieved Documents:

--- Result 1 ---
## Other HR Notes - Maxine participated in various company-sponsored trainings related...

Chunk 10 from employees.

## Other HR Notes
- Maxine participated in various company-sponsored trainings related to big data technologies and cloud infrastructure.  
- She was recognized for her contributions with the prestigious Insurellm IIOTY Innovator Award in 2023.  
- Maxine is currently involved in the women-in-tech initiative and participates in mentorship programs to guide junior employees.

--- Result 2 ---
- **January 2021 - Present**: **Senior Data Engineer** * Maxine was promoted...

Chunk 4 from employees.

- **January 2021 - Present**: **Senior Data Engineer**  
  * Maxine was prom

In [54]:
test = tests[0]

candidates = fetch_context_unranked(
    test.question,
    k=50,
)

retrieved = rerank(
    test.question,
    candidates,
)[:10]

for keyword in test.keywords:
    found = any(
        keyword.lower() in doc.page_content.lower()
        for doc in retrieved
    )

    print(f"{keyword}: {'FOUND' if found else 'NOT FOUND'}")

Maxine: FOUND
Thompson: NOT FOUND
IIOTY: FOUND


In [55]:
test = tests[0]

docs = fetch_context_unranked(
    test.question,
    k=10,
)

print("BEFORE RERANK")
for i, doc in enumerate(docs):
    print(f"\n--- {i+1} ---")
    print(doc.page_content[:300])

retrieved = rerank(
    test.question,
    docs,
)

print("\n\nAFTER RERANK")
for i, doc in enumerate(retrieved):
    print(f"\n--- {i+1} ---")
    print(doc.page_content[:300])

BEFORE RERANK

--- 1 ---
## Annual Performance History - **2023:** Rating: 4.9/5 *Exceptional performance. Led breakthrough...

Chunk 4 from employees.

## Annual Performance History
- **2023:** Rating: 4.9/5
  *Exceptional performance. Led breakthrough AI-driven underwriting project. Excellent technical leadership and ment

--- 2 ---
- **Awards**: - Insurellm "SDR of the Year" Award (2022) - Monthly...

Chunk 7 from employees.

- **Awards**:  
  - Insurellm "SDR of the Year" Award (2022)  
  - Monthly MVP Recognition (3 times in 2023)  

- **Interests**:  
  - In Alex's spare time, they enjoy participating in community volunteer

--- 3 ---
## Annual Performance History - **2023:** Rating: 4.9/5 *Outstanding performance. Instrumental in...

Chunk 4 from employees.

## Annual Performance History
- **2023:** Rating: 4.9/5
  *Outstanding performance. Instrumental in closing $3M in new business. Excellent technical presentations and client

--- 4 ---
## Annual Performance History - **2023:

In [56]:
print("Vectors in Chroma:", collection.count())

Vectors in Chroma: 884


In [57]:
query_embedding = embedding_model.encode(
    "IIOTY",
    normalize_embeddings=True
).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=10,
    include=["documents", "metadatas", "distances"]
)

print("Results:", len(results["documents"][0]))

for i, doc in enumerate(results["documents"][0]):
    print(f"\n--- {i+1} ---")
    print(doc[:1000])

Results: 10

--- 1 ---
- **2017:** Rating: 4.1/5 *Good performance. Expanding expertise across full stack and...

Chunk 5 from employees.

- **2017:** Rating: 4.1/5
  *Good performance. Expanding expertise across full stack and taking on more complex features.*

- **2016:** Rating: 4.0/5
  *Strong start at Insurellm. Quick to learn domain and contribute effectively.*

--- 2 ---
- **Recognition:** Data Science Excellence Award 2023, featured in InsureTech Innovation Magazine...

Chunk 8 from employees.

- **Recognition:** Data Science Excellence Award 2023, featured in InsureTech Innovation Magazine
- **Skills:** Expert in Python, TensorFlow, PyTorch, scikit-learn, SQL, and cloud ML platforms
- **Feedback:** World-class technical talent with strong business acumen. Natural leader who elevates entire team. Key retention priority.

--- 3 ---
## Annual Performance History - **2023:** Rating: 4.6/5 *Exceptional year with successful...

Chunk 3 from employees.

## Annual Performance History


In [58]:
print(tests[0])

question='Who won the prestigious IIOTY award in 2023?' keywords=['Maxine', 'Thompson', 'IIOTY'] reference_answer='Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023.' category='direct_fact'


In [59]:
results = collection.query(
    query_embeddings=[
        embedding_model.encode(
            "Maxine Thompson IIOTY award 2023",
            normalize_embeddings=True
        ).tolist()
    ],
    n_results=10,
    include=["documents", "metadatas", "distances"]
)

for i, doc in enumerate(results["documents"][0]):
    print(f"\n--- Result {i+1} ---")
    print(doc[:1000])


--- Result 1 ---
## Other HR Notes - Maxine participated in various company-sponsored trainings related...

Chunk 10 from employees.

## Other HR Notes
- Maxine participated in various company-sponsored trainings related to big data technologies and cloud infrastructure.  
- She was recognized for her contributions with the prestigious Insurellm IIOTY Innovator Award in 2023.  
- Maxine is currently involved in the women-in-tech initiative and participates in mentorship programs to guide junior employees.

--- Result 2 ---
- **January 2021 - Present**: **Senior Data Engineer** * Maxine was promoted...

Chunk 4 from employees.

- **January 2021 - Present**: **Senior Data Engineer**  
  * Maxine was promoted to Senior Data Engineer after successfully leading a pivotal project that improved data retrieval times by 30%. She now mentors junior engineers and is involved in strategic data initiatives, solidifying her position as a valued asset at Insurellm. She was recognized as Insurellm In

In [60]:
# 25. Evaluate the first test.

if tests:

    retrieval_result = evaluate_retrieval(
        tests[0]
    )

    print(retrieval_result)

mrr=0.6666666666666666 ndcg=0.6666666666666666 keywords_found=2 total_keywords=3 keyword_coverage=0.6666666666666666


In [61]:
# 26. Free answer evaluation.
#
# The original Day 4 answer evaluator used an OpenAI/LiteLLM judge.
# This replacement uses the reference answer + required keywords locally,
# so there is no external API.

class AnswerEval(BaseModel):
    feedback: str
    accuracy: float
    completeness: float
    relevance: float


def evaluate_answer(test):

    generated_answer, retrieved = answer_question(
        test.question
    )

    answer_lower = generated_answer.lower()

    found = [
        keyword
        for keyword in test.keywords
        if keyword.lower()
        in answer_lower
    ]

    missing = [
        keyword
        for keyword in test.keywords
        if keyword.lower()
        not in answer_lower
    ]

    coverage = (
        len(found) / len(test.keywords)
        if test.keywords
        else 0.0
    )

    if coverage == 1:
        accuracy = 5.0
        completeness = 5.0
    elif coverage >= 2 / 3:
        accuracy = 4.0
        completeness = 4.0
    elif coverage > 0:
        accuracy = 3.0
        completeness = 3.0
    else:
        accuracy = 1.0
        completeness = 1.0

    question_terms = tokenize(
        test.question
    )

    answer_terms = tokenize(
        generated_answer
    )

    overlap = (
        len(question_terms & answer_terms)
        / len(question_terms)
        if question_terms
        else 0.0
    )

    if overlap >= 0.50:
        relevance = 5.0
    elif overlap >= 0.30:
        relevance = 4.0
    elif overlap >= 0.10:
        relevance = 3.0
    else:
        relevance = 1.0

    if missing:
        feedback = (
            "Missing required keywords: "
            + ", ".join(missing)
        )
    else:
        feedback = (
            "All required keywords were found "
            "in the generated answer."
        )

    result = AnswerEval(
        feedback=feedback,
        accuracy=accuracy,
        completeness=completeness,
        relevance=relevance,
    )

    return (
        result,
        generated_answer,
        retrieved,
    )

In [62]:
# 27. Evaluate the first answer.

if tests:

    answer_eval, generated, retrieved = (
        evaluate_answer(tests[0])
    )

    print("Generated answer:")
    print(generated)

    print("\nFeedback:")
    print(answer_eval.feedback)

    print("\nScores:")
    print("Accuracy:", answer_eval.accuracy)
    print("Completeness:", answer_eval.completeness)
    print("Relevance:", answer_eval.relevance)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated answer:
Insurellm "SDR of the Year" Award (2022)
Monthly MVP Recognition (3 times in 2023)

I don't know based on the available knowledge base.

Feedback:
Missing required keywords: Maxine, Thompson, IIOTY

Scores:
Accuracy: 1.0
Completeness: 1.0
Relevance: 5.0


In [63]:
# 28. Evaluate all retrieval tests.

if tests:

    retrieval_results = [
        evaluate_retrieval(test)
        for test in tests
    ]

    avg_mrr = (
        sum(r.mrr for r in retrieval_results)
        / len(retrieval_results)
    )

    avg_ndcg = (
        sum(r.ndcg for r in retrieval_results)
        / len(retrieval_results)
    )

    avg_coverage = (
        sum(
            r.keyword_coverage
            for r in retrieval_results
        )
        / len(retrieval_results)
    )

    print("Tests:", len(tests))
    print("Average MRR:", round(avg_mrr, 4))
    print("Average nDCG:", round(avg_ndcg, 4))
    print(
        "Average keyword coverage:",
        round(avg_coverage, 2),
        "%"
    )

Tests: 150
Average MRR: 0.8154
Average nDCG: 0.8144
Average keyword coverage: 0.93 %


In [64]:
# 29. Final health check.

print("=" * 60)
print("FINAL CHECK")
print("=" * 60)

print("Knowledge-base files:", len(md_files))
print("Documents:", len(documents))
print("Chunks:", len(chunks))
print("Chroma vectors:", collection.count())
print("Embedding dimensions:", vectors.shape[1])
print("Embedding model:", EMBEDDING_MODEL)
print("Generation model:", GENERATION_MODEL)
print("OpenAI required: NO")
print("OpenAI API key required: NO")
print("LiteLLM required: NO")
print("=" * 60)

FINAL CHECK
Knowledge-base files: 76
Documents: 76
Chunks: 884
Chroma vectors: 884
Embedding dimensions: 384
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Generation model: Qwen/Qwen2.5-0.5B-Instruct
OpenAI required: NO
OpenAI API key required: NO
LiteLLM required: NO
